# 06 · Serving and monitoring — the model as a running system

**The question this notebook answers:** what happens between "a request
arrives" and "a decision leaves", what the service refuses to do, and how we
would notice the world changing underneath it.

Everything here runs **in-process**: FastAPI's `TestClient` sends real HTTP
requests to the app object without starting a server. It is the same app the
Docker container runs.

## 0 · Setup

In [1]:
import os, sys, json
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from IPython.display import Image
from config.constants import TABLES_DIR, FIGURES_DIR, DATA_PROCESSED, ARTIFACTS_DIR
pd.set_option("display.width", 140)

## 1 · Where the service gets its model: the hash-gated bundle

**What:** `artifacts/` holds exactly what serving needs — 5 fold models, the
calibrator, the feature list, the categorical contract, golden test cases,
the monitoring reference — and `MANIFEST.json` records a SHA-256 fingerprint
of each file.

**Why:** the Docker build, CI and this notebook all check those fingerprints.
If any file changes by one byte without a rebuild, the check fails *before* a
silently different model can serve decisions.

**Who:** the API and the demo load **only** from here
(`DeployedPipeline.from_artifacts()`).

In [2]:
import hashlib
manifest = json.load(open(os.path.join(ARTIFACTS_DIR, "MANIFEST.json")))["sha256"]
ok = {f: hashlib.sha256(open(os.path.join(ARTIFACTS_DIR, f), "rb").read()).hexdigest() == h
      for f, h in manifest.items()}
print(f"{sum(ok.values())}/{len(ok)} artifact hashes verified")

15/15 artifact hashes verified


## 2 · One request, end to end

**What:** score one applicant through `POST /score`. The request is a
**precomputed feature vector** — the service does not re-run the bureau
joins; in production that is an upstream feature pipeline's job.

**How to read the response:** besides the decision it returns *every input
to the decision* — threshold, LGD, EAD, margin, cure rate, the term used —
plus three reason codes and the model version, so any decision can be
audited later.

In [3]:
import tempfile
# this notebook logs to its own scratch file, never the service's request log
os.environ["CREDIT_RISK_REQUEST_LOG"] = os.path.join(tempfile.mkdtemp(), "notebook_log.sqlite")

from fastapi.testclient import TestClient
from src.api.app import API_FIELDS, CATS, app

client = TestClient(app)
rows = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "golden_inputs.parquet"))

def payload(row):
    return {f: (None if pd.isna(row[f]) else (str(row[f]) if f in CATS else float(row[f])))
            for f in API_FIELDS}

base = payload(rows.iloc[0])
response = client.post("/score", json=base).json()
{k: v for k, v in response.items() if k != "reason_codes"}

/home/claude/venv312/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'pd_calibrated': 0.011682242990654205,
 'decision': 'APPROVE',
 'threshold_applied': 0.10694767669900375,
 'lgd_used': 0.7,
 'ead_factor_used': 0.85,
 'margin_used': 0.07125435540069687,
 'cure_rate_used': 0.0,
 'term_years_used': 2.8501742160278747,
 'term_fallback_used': False,
 'score_scaled': 707.7329122394501,
 'unseen_category_counts': {'ORGANIZATION_TYPE': 0,
  'OCCUPATION_TYPE': 0,
  'NAME_EDUCATION_TYPE': 0,
  'NAME_FAMILY_STATUS': 0,
  'NAME_CONTRACT_TYPE': 0,
  'WALLSMATERIAL_MODE': 0,
  'NAME_INCOME_TYPE': 0,
  'NAME_HOUSING_TYPE': 0},
 'cost_overrides_used': {},
 'model_version': '69ef259f612a',
 'note': "cost parameters are illustrative economic assumptions; monetary amounts are in the dataset's unspecified currency units, not euros"}

## 3 · What the service refuses — and why each rule exists

A scoring service must not quietly score nonsense. Each probe below
reproduces a defect found in the October 2026 review.

1. **Client-chosen loan term.** `term_years` drives both the model and the
   threshold. A 43.8%-PD applicant flipped from REJECT to APPROVE by sending
   `term_years = 7`. Now the server *derives* term from
   `AMT_CREDIT / (12 × AMT_ANNUITY)` and refuses a contradicting value.
2. **Near-empty request.** Only the loan amount and contract type → the model
   would follow its "missing" branches and approve at 2.2%. No training
   applicant was ever that empty, so it is refused as outside the training
   support.
3. **Unknown fields.** Including `CODE_GENDER`: the protected attribute cannot
   even reach the scoring code.
4. **Economics outside the frozen grids.** The demo can explore
   sensitivity, but only inside the pre-registered ranges.

In [4]:
probes = {
    "term_years = 7 (contradicts the amounts)": {**base, "term_years": 7.0},
    "near-empty payload": {"AMT_CREDIT": 1000.0, "NAME_CONTRACT_TYPE": "Cash loans"},
    "CODE_GENDER supplied": {**base, "CODE_GENDER": "M"},
    "LGD 0.99 (outside frozen grid)": {**base, "cost_overrides": {"lgd_cash": 0.99}},
    "cure rate 30% (inside grid)": {**base, "cost_overrides": {"cure_rate": 0.30}},
}
for name, body in probes.items():
    r = client.post("/score", json=body)
    detail = r.json().get("detail") if r.status_code != 200 else f"t* = {r.json()['threshold_applied']:.4f}"
    print(f"{r.status_code}  {name:<42} {str(detail)[:90]}")

422  term_years = 7 (contradicts the amounts)   inconsistent derived features (they are recomputed server-side from their sources): term_y
422  near-empty payload                         32 of 36 application-side features missing; no training applicant had more than 12. Refusi
422  CODE_GENDER supplied                       [{'type': 'extra_forbidden', 'loc': ['body', 'CODE_GENDER'], 'msg': 'Extra inputs are not 
422  LGD 0.99 (outside frozen grid)             lgd_cash=0.99 outside the frozen sensitivity grid [0.6, 0.8]
200  cure rate 30% (inside grid)                t* = 0.1461


## 4 · Monitoring: noticing that the world changed

**The problem, from scratch:** a credit model is trained on the past. If the
economy turns, applicants start to look different, and the model's numbers
quietly stop meaning what they used to. We cannot wait for defaults to show it
— those arrive 12+ months later.

**What we watch instead — PSI (population stability index):** compare the
distribution of a feature (or of the score) today against the development
reference. PSI below 0.10 = stable, 0.10–0.25 = investigate, above 0.25 =
retrain trigger (Siddiqi 2006).

**What we simulate:** a severe recession on 1,000 applicants — external
credit scores −0.10, payment burden +25%. This is a **simulated** scenario to
prove the alarm works, not production telemetry.

**Where:** `src.monitoring.monitoring_report` and `simulate_recession`.

In [5]:
from src.models.pipeline import DeployedPipeline
from src.monitoring import load_reference, monitoring_report, simulate_recession

pool = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "demo_pool.parquet")).sample(1000, random_state=1)
pipe = DeployedPipeline.from_artifacts()
ref = load_reference()
out = {}
for name, frame in (("baseline", pool), ("SIMULATED recession", simulate_recession(pool))):
    s = pipe.score_frame(frame, with_reason_codes=False)
    rep = monitoring_report(frame, s.pd_raw_ensemble.values, s.threshold_applied.values,
                            s.decision.values, ref)
    out[name] = {"score PSI": rep["score_psi"]["psi"],
                 "worst feature PSI": max(v["psi"] for v in rep["feature_psi"].values()),
                 "approval rate": rep["approval_rate"],
                 "mean t*(x)": rep["mean_threshold_t_star"]}
pd.DataFrame(out).round(3)

,baseline,SIMULATED recession
score PSI,0.011,0.321
worst feature PSI,0.078,0.652
approval rate,0.625,0.444
mean t*(x),0.080,0.080


**How to read it:** under the simulated recession the worst feature PSI jumps
into the retrain band and approvals fall — the alarm fires. The mean
threshold does **not** move: thresholds depend on contract type and term, not
on credit scores. If the *product mix* shifted instead (say, more short
loans), mean `t*(x)` would move with no score drift at all — that is why both
are tracked.

## 5 · The live request log

**What:** every `/score` call above was logged (prediction, threshold,
decision — no applicant features, which would be personal data). This
notebook pointed the log at a scratch file in section 2, so only its own
requests appear here.
`read_log()` turns it into rolling approval-rate and mean-threshold series;
the Streamlit Monitoring tab plots them.

In [6]:
from src.monitoring import read_log
log = read_log(window=5)
print(f"{len(log)} requests logged")
log.tail()[["ts", "p_cal", "threshold", "decision", "rolling_approval_rate", "rolling_mean_t_star"]]

2 requests logged


,ts,p_cal,threshold,decision,rolling_approval_rate,rolling_mean_t_star
0,2026-10-07T13:21:20.066473,0.011682,0.106948,APPROVE,1.0,0.106948
1,2026-10-07T13:21:20.249701,0.011682,0.146087,APPROVE,1.0,0.126517
